# Requête HTTP 

Un requête HTTP est une requête basée sur le protocole TCP, elle fait partie de la couche application de la couche OSI. Elle permet d'accéder aux données mise à disposition sur une adresse IP (ou url résolue par un DNS) et un port. 

Les deux ports les plus utilisés dans le web sont le 80 pour les sites en HTTP et le 443 pour les sites en HTTPS. HTTPS est une variable du protocole HTTP basé sur le protocole TLS.

Il existe de nombreux types de requêtes selon la convention `REST`: 
- GET
- POST
- PUT 
- DELETE
- UPDATE.

Dans notre cas, nous allons utiliser la plupart du temps des GET et potentiellement des POST. 
- Le GET permet comme son nom l'indique de récupérer des informations en fonction de certains paramètres. 
- Le POST nécessite un envoi de données pour récupérer des données. Le body du post est, la plupart du temps, envoyé sous la forme d'un objet JSON.

Ces requêtes encapsulent un certain nombre de paramètres qui permettent soient d'identifier une provenance et un utilisateur ou de réaliser différentes actions.

In [ ]:
import requests

In [ ]:
url = "https://www.esiee.fr/"
response = requests.get(url)
response.status_code

Il existe deux méthodes pour récupérer le contenu de la page :

- `response.text` qui permet de retourner le texte sous la forme d'une chaine de charactères.
- `response.content` qui permet de récupérer le contenu de la page sous la forme de bytes

In [ ]:
type(response.content)

In [ ]:
type(response.text)

Pour récupérer les 1000 premiers charactères de la page :

In [ ]:
response.text[0:1000]

Pour récupérer les headers HTTP de la réponse :

In [ ]:
response.headers

On peut modifier les paramètres de la requête et/ou ses headers. On peut par exemple ajouter un UserAgent (identifiant de l'initiateur de la requête) et un timeout de 10 secondes :

In [ ]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
response = requests.get(url, headers=headers, timeout = 10)
response.content[0:1000]

## Exercice

## Exercice 1

- Créer une classe Python permettant de faire des requêtes HTTP.
- Cette classe doit utiliser toujours le même UserAgent.
- Le TimeOut sera spécifié à chaque appelle avec une valeur par défaut.
- Un mécanisme de retry sera mis en place de façon recursive.

## Exercice 2

- Faire une fonction permettant de supprimer tous les espaces supperflus d'une string
- Faire une fonction qui prend une string html et renvois une string intelligible (enlever les caractères spéciaux,
- Récupérer le domaine en fonction d'un url

In [ ]:
import requests
import time


class HTTPClient:

    def __init__(self):
        self.user_agent = "DataEngineeringStudent/1.0"

    def get(self, url, timeout=10, retries=3):
        headers = {"User-Agent": self.user_agent}

        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=timeout
            )
            response.raise_for_status()
            return response

        except requests.RequestException:
            if retries > 0:
                time.sleep(1)
                return self.get(url, timeout, retries - 1)
            raise

In [ ]:
client = HTTPClient()

response = client.get("https://www.esiee.fr/")

print(response.status_code)
print(response.text[:500])

In [ ]:
import html
from bs4 import BeautifulSoup
from urllib.parse import urlparse


def clean_spaces(text):
    return " ".join(text.split())


def clean_html(text):
    soup = BeautifulSoup(html.unescape(text), "html.parser")
    return clean_spaces(soup.get_text(" "))


def get_domain(url):
    return urlparse(url).hostname

In [ ]:
print(clean_spaces("Bonjour     tout    le monde"))

print(clean_html("<p>Bonjour &amp; bienvenue</p>"))

print(get_domain("https://www.esiee.fr/formations"))

# Exploitation du HTML  

Ici, il faut récupérer le code HTML d'un site web à partir d'une requête. Lorsque vous avez récupéré le texte d'un site il faut le parser. Pour cela, on utilise BeautifulSoup qui permet de transformer la structure HTML en objet Python. Cela permet de récupérer efficacement les données qui nous intéresse.

Pour les webmasters, le blocage le plus souvent mis en place et un blocage sur le User-Agent. Le User-Agent est un paramètre intégré dans la requête HTTP réalisé par le Navigateur pour envoyer au front des informations basiques :

- la version du Navigateur,
- la version de l'OS
- Le type de gestionnaire graphique (Gecko)
- le type de device utilisé

Exemple de User Agent :  

`Mozilla/5.0 (Windows NT 6.1; Win64; x64; rv:47.0) Gecko/20100101 Firefox/47.0`

Commençons à utiliser `BeautifulSoup`, il est normalement déjà installé, le cas échéant executez les lignes suivantes : 

In [ ]:
import requests
from bs4 import BeautifulSoup

Pour transformer une requête (requests) en objet BeautifulSoup :

In [ ]:
response = requests.get(url)
soup = BeautifulSoup(response.text)

Pour trouver tous les liens d'une page, on récupère la balise `a` qui permet de gérer les liens en HTML :

In [ ]:
soup.find_all("a")[0:10]

On peut aussi préciser la classe HTML qu'on veut récupérer :

```python
soup.find_all(class_="<CLASS_NAME>")[0:10]
```

Ici par exemple: 

In [ ]:
soup.find_all(class_="toggler")[0:5]

Pour récupérer le text sans les balises HTML :

In [ ]:
soup.text[0:1000]

## Exercice
### Exercice 3

Améliorer la classe développé précédemment.

- Ajouter une méthode pour récupérer l'objet soup d'un url
- Récupérer une liste de User Agent et effectuer une rotation aléatoire sur celui à utiliser
- Utiliser cette classe pour parser une page HTML et récupérer : le titre, tous les H1 (si ils existent), les liens vers les images, les liens sortants vers d'autres sites, et le texte principal.

In [ ]:
import random
from urllib.parse import urljoin


class WebScraper(HTTPClient):

    def __init__(self):
        self.user_agents = [
            "DataEngineeringStudent/1.0",
            "DataEngineeringStudent/1.1",
            "DataEngineeringStudent/1.2"
        ]

    def get(self, url, timeout=10, retries=3):
        self.user_agent = random.choice(self.user_agents)
        return super().get(url, timeout, retries)

    def get_soup(self, url):
        response = self.get(url)
        return BeautifulSoup(response.text, "html.parser")

    def parse_page(self, url):
        soup = self.get_soup(url)

        title = soup.title.get_text(strip=True) if soup.title else ""

        h1 = [
            tag.get_text(strip=True)
            for tag in soup.find_all("h1")
        ]

        images = [
            urljoin(url, img["src"])
            for img in soup.find_all("img", src=True)
        ]

        external_links = []

        for link in soup.find_all("a", href=True):
            full_url = urljoin(url, link["href"])

            if urlparse(full_url).scheme in ("http", "https"):
                if get_domain(full_url) != get_domain(url):
                    external_links.append(full_url)

        for tag in soup.find_all(
            ["script", "style", "nav", "footer", "header"]
        ):
            tag.decompose()

        main = soup.find("article") or soup.find("main") or soup.body

        text = clean_spaces(main.get_text(" ")) if main else ""

        return {
            "title": title,
            "h1": h1,
            "images": images,
            "external_links": external_links,
            "text": text
        }

In [ ]:
scraper = WebScraper()

data = scraper.parse_page("https://www.esiee.fr/")

print("Titre :", data["title"])
print("H1 :", data["h1"])
print("Images :", data["images"][:5])
print("Liens externes :", data["external_links"][:5])
print("Texte :", data["text"][:1000])

# Exploitation des appels d'API



Losque le front du site récupère des données sur une API gérée par le back, un appel d'API est réalisé. Cet appel est recensé dans les appels réseaux. Il est alors possible de re-jouer cet appel pour récupérer à nouveau les données. Il est très facile de récupérer ces appels dans l'onglet Network de la console développeur de Chrome ou FireFox. La console vous permet de copier le code CURL de la requête et vous pouvez ensuite la transformer en code Python depuis le site https://curl.trillworks.com/.

Souvent les APIs sont bloquées avec certains paramètres. L'API vérifie que dans les headers de la requête HTTP ces paramètres sont présents :
* un token généré à la volée avec des protocoles OAuth2 (ou moins développés).
* un referer provenant du site web (la source de la requête), très facile à falsifier.



## Exercice 
### Exercice 4

- Utiliser les informations développées plus haut pour récupérer les premiers résultats d'une recherche d'une requête
sur Google. 

Tips : 

- Ouvrir les outils de développements de Chrome ou Firefox
- Onglet Network
- Fouiller dans les requêtes pour voir à quoi ressemble un appel API Google
- Utilisez beautiful soup pour convertir le contenu de la request en objet et accéder aux balises

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://www.google.com/search"

params = {"q": "esiee paris"}

response = requests.get(url, params=params, timeout=10)

print("Code HTTP :", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

results = soup.find_all("h3")

for result in results[:5]:
    print(result.get_text(strip=True))

if not results:
    print("Aucun résultat accessible.")

In [ ]:
print(response.url)
print(soup.title)
print(soup.get_text(" ", strip=True)[:1500])

In [ ]:
for link in soup.find_all("a", href=True)[:15]:
    print(link.get_text(strip=True), link["href"])

### Conclusion de l'exercice 4

La requête HTTP retourne un code 200, mais Google
redirige vers une page de consentement aux cookies.

Il est donc impossible de récupérer les résultats
de recherche à partir du HTML obtenu.

# Exercice Final  

Exercice Final
Utilisez tout ce que vous avez appris pour récupérer des articles de News avec une catégorie. Il est souvent intéressant de partir des flux RSS pour commencer :

Les données doivent comprendre :
- Le texte important propre
- L'url
- Le domaine
- la catégorie
- Le titre de l'article
- Le titre de la page
- (Facultatif) : les images

Tips : 

- Taper le nom de votre média favoris + RSS (par exemple : https://www.lemonde.fr/rss/)
- Aller dans le DOM de la page 
- Trouver les catégories et les liens vers les articles

In [ ]:
import requests
import xml.etree.ElementTree as ET

rss_url = "https://feeds.bbci.co.uk/news/technology/rss.xml"

response = requests.get(rss_url, timeout=15)
response.raise_for_status()

root = ET.fromstring(response.content)

for item in root.findall(".//item")[:5]:
    print("Titre :", item.findtext("title"))
    print("URL :", item.findtext("link"))
    print()

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import pandas as pd
import time

articles = []

for item in root.findall(".//item")[:5]:

    url = item.findtext("link")
    titre = item.findtext("title")

    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        article = soup.find("article") or soup.find("main")

        paragraphes = article.find_all("p") if article else []

        texte = " ".join(
            p.get_text(strip=True) for p in paragraphes
        )

        texte = " ".join(texte.split())

        articles.append({
            "texte": texte,
            "url": url,
            "domaine": urlparse(url).hostname,
            "categorie": "Technologie",
            "titre_article": titre,
            "titre_page": soup.title.get_text(strip=True) if soup.title else ""
        })

    except requests.RequestException as e:
        print("Erreur :", e)

    time.sleep(1)

df = pd.DataFrame(articles)

df

In [ ]:
print("Nombre d'articles :", len(df))
print("Colonnes :", df.columns.tolist())

### Conclusion de l'exercice final

Nous avons récupéré cinq articles de la BBC dans la
catégorie Technologie à partir d'un flux RSS.

Les données ont été extraites avec requests et
BeautifulSoup, puis structurées dans un DataFrame Pandas.

Les six colonnes demandées ont été obtenues.